In [1]:
import pandas as pd, numpy as np
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 60)

RAW  = Path(r"C:\Users\Samiksha gupta\Downloads\raw-20261024T102850Z-1-001\raw")   # 13 .xlsx files yahan hain
PROC = Path(r"C:\Users\Samiksha gupta\Downloads\processed")                        # cleaned files yahan save hongi
PROC.mkdir(parents=True, exist_ok=True)

In [3]:
names = ['campaigns','channels','calendar','customers','products','orders','order_items',
         'carts','web_sessions','ad_spend_daily','crm_messages','social_post_daily','lead_events']
raw = {n: pd.read_excel(RAW / f'{n}.xlsx') for n in names}
for n, x in raw.items():
    print(f'{n:20s} rows={x.shape[0]:>6}  cols={x.shape[1]}')

campaigns            rows=    11  cols=7
channels             rows=     9  cols=4
calendar             rows=   365  cols=7
customers            rows= 10000  cols=11
products             rows=     8  cols=5
orders               rows= 13295  cols=19
order_items          rows= 17284  cols=9
carts                rows= 38260  cols=12
web_sessions         rows= 76389  cols=11
ad_spend_daily       rows=  3160  cols=7
crm_messages         rows= 23670  cols=10
social_post_daily    rows=  1260  cols=14
lead_events          rows=  7406  cols=7


In [4]:
audit = pd.DataFrame([{'table': n, 'rows': len(x), 'missing_cells': int(x.isna().sum().sum()),
                       'duplicate_rows': int(x.duplicated().sum())} for n, x in raw.items()])
print(audit.to_string(index=False))

print('\nColumns with missing values:')
for n, x in raw.items():
    m = x.isna().sum(); m = m[m > 0]
    if len(m): print(f'  {n}: {m.to_dict()}')

# Primary keys must be unique
pk = {'campaigns':'campaign_id','channels':'channel_id','customers':'customer_id','products':'product_id',
      'orders':'order_id','order_items':'order_item_id','carts':'cart_id','web_sessions':'session_id',
      'ad_spend_daily':'spend_id','crm_messages':'message_id','social_post_daily':'social_record_id','lead_events':'lead_id'}
print('\nDuplicate primary keys:')
for t, k in pk.items(): print(f'  {t}.{k}: {raw[t][k].duplicated().sum()}')

# Foreign keys must exist in the parent table
def fk(child, col, parent, key):
    v = raw[child][col].dropna()
    print(f'  {child}.{col} -> {parent}.{key}: unmatched = {(~v.isin(raw[parent][key])).sum()}')
print('\nBroken links:')
fk('orders','customer_id','customers','customer_id');      fk('orders','campaign_id','campaigns','campaign_id')
fk('order_items','order_id','orders','order_id');          fk('order_items','product_id','products','product_id')
fk('carts','session_id','web_sessions','session_id');      fk('carts','order_id','orders','order_id')
fk('web_sessions','channel_id','channels','channel_id');   fk('web_sessions','campaign_id','campaigns','campaign_id')
fk('customers','acquisition_campaign_id','campaigns','campaign_id')
fk('ad_spend_daily','campaign_id','campaigns','campaign_id')
fk('crm_messages','customer_id','customers','customer_id')

            table  rows  missing_cells  duplicate_rows
        campaigns    11              0               0
         channels     9              0               0
         calendar   365              0               0
        customers 10000              0               0
         products     8              0               0
           orders 13295          14513               0
      order_items 17284              0               0
            carts 38260          50503               0
     web_sessions 76389          64991               0
   ad_spend_daily  3160              0               0
     crm_messages 23670              0               0
social_post_daily  1260              0               0
      lead_events  7406           3270               0

Columns with missing values:
  orders: {'campaign_id': 1609, 'refund_date': 12904}
  carts: {'customer_id': 22733, 'order_id': 27770}
  web_sessions: {'customer_id': 41881, 'campaign_id': 23110}
  lead_events: {'customer_id': 307

In [10]:
d = {n: x.copy() for n, x in raw.items()}      # work on copies, keep raw untouched

# 4a. Strip hidden spaces from text columns
for n, x in d.items():
    for c in x.columns:
        if x[c].dtype == 'object' or pd.api.types.is_string_dtype(x[c]):
            try: x[c] = x[c].str.strip()
            except AttributeError: pass

# 4b. Convert date columns to real dates
date_cols = {'calendar':['date','month'], 'customers':['customer_created_date'],
             'orders':['order_date','promised_delivery_date','delivered_date','refund_date'],
             'carts':['created_date'], 'web_sessions':['session_date'], 'ad_spend_daily':['date'],
             'crm_messages':['sent_date'], 'social_post_daily':['date'],
             'lead_events':['captured_date'], 'campaigns':['start_date','end_date']}
for t, cols in date_cols.items():
    for c in cols:
        before = d[t][c].isna().sum()
        d[t][c] = pd.to_datetime(d[t][c], errors='coerce')
        if d[t][c].isna().sum() > before:
            print(f'WARNING {t}.{c}: {d[t][c].isna().sum()-before} unreadable dates')

# short names for convenience
campaigns, channels, calendar, customers, products = d['campaigns'], d['channels'], d['calendar'], d['customers'], d['products']
orders, order_items, carts, web_sessions = d['orders'], d['order_items'], d['carts'], d['web_sessions']
ad_spend, crm, social, leads = d['ad_spend_daily'], d['crm_messages'], d['social_post_daily'], d['lead_events']

In [11]:
def check(label, mask): print(f'{label:58s}: {int(mask.sum())}')

check('orders: delivered before order date',            orders.delivered_date < orders.order_date)
check('orders: promised date before order date',        orders.promised_delivery_date < orders.order_date)
check('orders: refund bigger than net revenue',         orders.refund_amount_inr > orders.net_product_revenue_inr)
check('orders: Returned but refund = 0',                (orders.order_status=='Returned') & (orders.refund_amount_inr==0))
check('orders: Delivered but refund > 0',               (orders.order_status=='Delivered') & (orders.refund_amount_inr>0))
check('orders: gross - discount != net',                (orders.gross_product_revenue_inr - orders.discount_amount_inr - orders.net_product_revenue_inr).abs() > 0.01)
check('order_items: qty x price != gross',              (order_items.quantity*order_items.unit_price_inr - order_items.gross_amount_inr).abs() > 0.01)
check('ad_spend: clicks > impressions',                 ad_spend.clicks > ad_spend.impressions)
check('ad_spend: media spend > marketing spend',        ad_spend.media_spend_inr > ad_spend.marketing_spend_inr)
check('social: reach > impressions',                    social.reach > social.impressions)
check('crm: clicked but not opened',                    (crm.clicked==1) & (crm.opened==0))
check('crm: opened but not delivered',                  (crm.opened==1) & (crm.delivered==0))
check('carts: Completed without order_id',              (carts.cart_status=='Completed') & carts.order_id.isna())
check('carts: Abandoned with an order_id',              (carts.cart_status=='Abandoned') & carts.order_id.notna())
chk = orders.merge(customers[['customer_id','customer_created_date']], on='customer_id')
check('orders placed before customer was created',      chk.order_date < chk.customer_created_date)

print('\nNegative values:')
for n, x in d.items():
    neg = (x.select_dtypes('number') < 0).sum(); neg = neg[neg > 0]
    if len(neg): print(f'  {n}: {neg.to_dict()}')

orders: delivered before order date                       : 0
orders: promised date before order date                   : 0
orders: refund bigger than net revenue                    : 0
orders: Returned but refund = 0                           : 0
orders: Delivered but refund > 0                          : 0
orders: gross - discount != net                           : 0
order_items: qty x price != gross                         : 0
ad_spend: clicks > impressions                            : 0
ad_spend: media spend > marketing spend                   : 0
social: reach > impressions                               : 0
crm: clicked but not opened                               : 0
crm: opened but not delivered                             : 0
carts: Completed without order_id                         : 0
carts: Abandoned with an order_id                         : 0
orders placed before customer was created                 : 0

Negative values:


In [12]:
def iqr_out(s):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return int(((s < q1-1.5*i) | (s > q3+1.5*i)).sum())

for t, c in [('web_sessions','page_load_seconds'), ('web_sessions','session_duration_seconds'),
             ('carts','cart_value_inr'), ('orders','net_product_revenue_inr'), ('orders','refund_amount_inr')]:
    s = d[t][c]
    print(f'{t}.{c}: min={s.min():.1f} median={s.median():.1f} max={s.max():.1f}  IQR-outliers={iqr_out(s)}')

web_sessions.page_load_seconds: min=0.7 median=3.5 max=8.3  IQR-outliers=83
web_sessions.session_duration_seconds: min=15.0 median=410.0 max=800.0  IQR-outliers=0
carts.cart_value_inr: min=170.0 median=499.0 max=4095.0  IQR-outliers=1308
orders.net_product_revenue_inr: min=136.0 median=510.0 max=3657.0  IQR-outliers=457
orders.refund_amount_inr: min=0.0 median=0.0 max=2913.3  IQR-outliers=391


In [16]:
o = orders.copy()
o['discount_pct'] = np.where(o.gross_product_revenue_inr > 0, o.discount_amount_inr / o.gross_product_revenue_inr, 0)
o['net_revenue_after_refund_inr'] = o.net_product_revenue_inr - o.refund_amount_inr
# contribution = what is left after product, fulfilment, delivery, platform fee and refunds (before marketing spend)
o['contribution_margin_inr'] = (o.net_product_revenue_inr - o.refund_amount_inr - o.product_cost_inr
                                - o.fulfilment_cost_inr - o.delivery_cost_inr - o.platform_fee_inr)
o['contribution_incl_shipping_inr'] = o.contribution_margin_inr + o.shipping_fee_inr
o['contribution_margin_pct'] = np.where(o.net_product_revenue_inr > 0, o.contribution_margin_inr / o.net_product_revenue_inr, np.nan)
o['gross_margin_pct'] = np.where(o.net_product_revenue_inr > 0, (o.net_product_revenue_inr - o.product_cost_inr) / o.net_product_revenue_inr, np.nan)
o['delay_days'] = (o.delivered_date - o.promised_delivery_date).dt.days
o['is_late'] = (o.delay_days > 0).astype(int)
o['is_refunded'] = (o.refund_amount_inr > 0).astype(int)
o['has_discount'] = (o.discount_amount_inr > 0).astype(int)
o['campaign_attributed'] = o.campaign_id.notna().astype(int)
o['order_month'] = o.order_date.dt.to_period('M').astype(str)
o = o.sort_values(['customer_id', 'order_date', 'order_id']).reset_index(drop=True)
o['order_number'] = o.groupby('customer_id').cumcount() + 1
o['is_first_order'] = (o.order_number == 1).astype(int)
o['days_since_first_order'] = (o.order_date - o.groupby('customer_id').order_date.transform('min')).dt.days

In [17]:
li = order_items.merge(products[['product_id','product_name','category']], on='product_id', how='left')
li['line_margin_inr'] = li.net_product_revenue_inr - li.product_cost_inr
li['line_margin_pct'] = np.where(li.net_product_revenue_inr > 0, li.line_margin_inr / li.net_product_revenue_inr, np.nan)

In [18]:
c = carts.copy()
c['discount_pct'] = np.where(c.cart_value_inr > 0, c.offered_discount_inr / c.cart_value_inr, 0)
c['is_converted'] = c.order_id.notna().astype(int)
c['abandon_stage'] = np.select(
    [c.cart_status == 'Completed', c.payment_attempted == 1, c.checkout_started == 1],
    ['Completed', 'Payment attempted - not completed', 'Checkout started - no payment'],
    default='Cart only - no checkout')
c['created_month'] = c.created_date.dt.to_period('M').astype(str)

In [19]:
cs = c.groupby('session_id').agg(n_carts=('cart_id','nunique'), reached_checkout=('checkout_started','max'),
                                 attempted_payment=('payment_attempted','max'), converted=('is_converted','max')).reset_index()
s = web_sessions.merge(cs, on='session_id', how='left')
s['has_cart'] = (s.n_carts.fillna(0) > 0).astype(int)
for col in ['reached_checkout','attempted_payment','converted']:
    s[col] = s[col].fillna(0).astype(int)
s = s.merge(channels[['channel_id','channel_name','channel_group']], on='channel_id', how='left')
s['is_mobile'] = (s.device_type == 'Mobile').astype(int)
s['load_speed_bucket'] = pd.cut(s.page_load_seconds, [0,2,3,4,5,np.inf],
                                labels=['<=2s','2-3s','3-4s','4-5s','>5s']).astype(str)
s['is_anonymous'] = s.customer_id.isna().astype(int)
s['campaign_missing'] = s.campaign_id.isna().astype(int)
s['session_month'] = s.session_date.dt.to_period('M').astype(str)

In [20]:
camp = campaigns.merge(channels[['channel_id','channel_name','channel_group']], on='channel_id', how='left')

ad = ad_spend.copy()
ad['month'] = ad.date.dt.to_period('M').astype(str)
ad['ctr'] = np.where(ad.impressions > 0, ad.clicks / ad.impressions, np.nan)
ad['cpc_inr'] = np.where(ad.clicks > 0, ad.marketing_spend_inr / ad.clicks, np.nan)

so = social.copy()
so['engagements'] = so.likes + so.comments + so.shares + so.saves
so['engagement_rate'] = np.where(so.reach > 0, so.engagements / so.reach, np.nan)
so['link_click_rate'] = np.where(so.reach > 0, so.link_clicks / so.reach, np.nan)
so['month'] = so.date.dt.to_period('M').astype(str)

ld = leads.copy()
ld['is_anonymous_lead'] = ld.customer_id.isna().astype(int)
ld['campaign_missing'] = ld.campaign_id.isna().astype(int)

In [21]:
m = crm.copy()
m['crm_channel'] = m.campaign_id.map(camp.set_index('campaign_id').channel_name)   # Email or WhatsApp
m['sent_month'] = m.sent_date.dt.to_period('M').astype(str)

In [22]:
snapshot = o.order_date.max()      # "today" for the dataset

og = o.groupby('customer_id').agg(
    first_order_date=('order_date','min'), last_order_date=('order_date','max'),
    total_orders=('order_id','nunique'), total_net_revenue_inr=('net_product_revenue_inr','sum'),
    avg_order_value_inr=('net_product_revenue_inr','mean'), total_contribution_inr=('contribution_margin_inr','sum'),
    avg_discount_pct=('discount_pct','mean'), total_refund_inr=('refund_amount_inr','sum'),
    late_orders=('is_late','sum')).reset_index()
second = o[o.order_number == 2].groupby('customer_id').order_date.min().rename('second_order_date').reset_index()

cust = customers.merge(og, on='customer_id', how='left').merge(second, on='customer_id', how='left')
cust['days_to_second_order'] = (cust.second_order_date - cust.first_order_date).dt.days
cust['repeat_any'] = cust.second_order_date.notna().astype(int)
rep90 = o.loc[o.days_since_first_order.between(1, 90), 'customer_id'].unique()
cust['repeat_90d'] = cust.customer_id.isin(rep90).astype(int)          # Problem 2 KPI, and the Phase 5 target
cust['observable_90d'] = (cust.first_order_date + pd.Timedelta(days=90) <= snapshot).astype(int)   # 1 = full 90 days visible
cust['recency_days'] = (snapshot - cust.last_order_date).dt.days
cust['tenure_days'] = (snapshot - cust.customer_created_date).dt.days
cust['cohort_month'] = cust.customer_created_date.dt.to_period('M').astype(str)

# acquisition campaign and channel names
acq = camp[['campaign_id','campaign_name','channel_name','channel_group']].rename(columns={
    'campaign_id':'acquisition_campaign_id','campaign_name':'acquisition_campaign',
    'channel_name':'acquisition_channel','channel_group':'acquisition_channel_group'})
cust = cust.merge(acq, on='acquisition_campaign_id', how='left')

# CRM engagement per customer (delivered messages only)
cm = m[m.delivered == 1].groupby('customer_id').agg(crm_delivered=('message_id','size'), crm_opened=('opened','sum'),
                                                    crm_clicked=('clicked','sum'), crm_unsubscribed=('unsubscribed','max')).reset_index()
cust = cust.merge(cm, on='customer_id', how='left')
for col in ['crm_delivered','crm_opened','crm_clicked','crm_unsubscribed']: cust[col] = cust[col].fillna(0).astype(int)
cust['crm_open_rate'] = np.where(cust.crm_delivered > 0, cust.crm_opened / cust.crm_delivered, 0)
cust['crm_click_rate'] = np.where(cust.crm_delivered > 0, cust.crm_clicked / cust.crm_delivered, 0)

# website behaviour per customer (known sessions only)
wb = s[s.customer_id.notna()].groupby('customer_id').agg(web_sessions=('session_id','nunique'),
                                                         avg_page_load_seconds=('page_load_seconds','mean')).reset_index()
ab = c[(c.customer_id.notna()) & (c.cart_status == 'Abandoned')].groupby('customer_id').size().rename('abandoned_carts').reset_index()
cust = cust.merge(wb, on='customer_id', how='left').merge(ab, on='customer_id', how='left')
cust['abandoned_carts'] = cust.abandoned_carts.fillna(0).astype(int)
cust['web_sessions'] = cust.web_sessions.fillna(0).astype(int)

In [23]:
a = ad.groupby(['campaign_id','month']).agg(marketing_spend_inr=('marketing_spend_inr','sum'), media_spend_inr=('media_spend_inr','sum'),
                                            impressions=('impressions','sum'), clicks=('clicks','sum')).reset_index()
r = (o.groupby(['campaign_id','order_month']).agg(orders=('order_id','nunique'), net_revenue_inr=('net_product_revenue_inr','sum'),
                                                  contribution_inr=('contribution_margin_inr','sum')).reset_index()
       .rename(columns={'order_month':'month'}))
n = (cust.groupby(['acquisition_campaign_id','cohort_month']).size().rename('new_customers').reset_index()
         .rename(columns={'acquisition_campaign_id':'campaign_id','cohort_month':'month'}))
cmth = a.merge(r, on=['campaign_id','month'], how='outer').merge(n, on=['campaign_id','month'], how='outer')
num = ['marketing_spend_inr','media_spend_inr','impressions','clicks','orders','net_revenue_inr','contribution_inr','new_customers']
cmth[num] = cmth[num].fillna(0)
cmth['roas'] = np.where(cmth.marketing_spend_inr > 0, cmth.net_revenue_inr / cmth.marketing_spend_inr, np.nan)
cmth['cac_inr'] = np.where((cmth.new_customers > 0) & (cmth.marketing_spend_inr > 0), cmth.marketing_spend_inr / cmth.new_customers, np.nan)
cmth['ctr'] = np.where(cmth.impressions > 0, cmth.clicks / cmth.impressions, np.nan)
cmth = cmth.merge(camp[['campaign_id','campaign_name','channel_name','channel_group','objective']], on='campaign_id', how='left')

# Whole-business monthly view. CAC uses acquisition campaigns only, so CRM spend is excluded.
acq_ids = campaigns.loc[campaigns.objective == 'Acquisition', 'campaign_id']
spend_m = ad[ad.campaign_id.isin(acq_ids)].groupby('month').marketing_spend_inr.sum().rename('acq_spend_inr')
newc_m  = cust.groupby('cohort_month').size().rename('new_customers')
rev_m   = o.groupby('order_month').agg(orders=('order_id','nunique'), net_revenue_inr=('net_product_revenue_inr','sum'),
                                       contribution_inr=('contribution_margin_inr','sum'))
monthly = pd.concat([spend_m, newc_m, rev_m], axis=1).fillna(0)
monthly.index.name = 'month'; monthly = monthly.reset_index()
monthly['cac_inr'] = np.where(monthly.new_customers > 0, monthly.acq_spend_inr / monthly.new_customers, np.nan)
print(monthly.round(1).to_string(index=False))

  month  acq_spend_inr  new_customers  orders  net_revenue_inr  contribution_inr  cac_inr
2025-10       486000.0          900.0     917         606684.2          168109.7    540.0
2025-11       534375.0          950.0    1049         646608.6          173205.4    562.5
2025-12       585000.0         1000.0    1186         734877.4          197131.2    585.0
2026-01       637875.0         1050.0    1321         842065.0          238487.2    607.5
2026-02       693000.0         1100.0    1398         906775.9          251458.5    630.0
2026-03       750375.0         1150.0    1496         973683.2          263539.9    652.5
2026-04       810000.0         1200.0    1587        1002323.0          276972.5    675.0
2026-05       871875.0         1250.0    1655        1012594.7          268626.2    697.5
2026-06      1008000.0         1400.0    1824        1130193.8          297878.6    720.0
2026-07            0.0            0.0     407         270376.5           80901.0      NaN
2026-08   

In [24]:
print('Item revenue = order revenue :', np.isclose(li.net_product_revenue_inr.sum(), o.net_product_revenue_inr.sum()))
print('Orders                       :', o.order_id.nunique())
print('Customers                    :', cust.customer_id.nunique())
print('Total net revenue (INR)      :', round(o.net_product_revenue_inr.sum()))
print('90-day repeat rate           :', round(cust.repeat_90d.mean()*100, 2), '%')
print('Cart abandonment rate        :', round((1 - c.is_converted.mean())*100, 2), '%')
print('Session -> order conversion  :', round(s.converted.mean()*100, 2), '%')
print('Blended CAC (acq. campaigns) :', round(monthly.acq_spend_inr.sum() / monthly.new_customers.sum(), 2))
print('Late delivery share          :', round(o.is_late.mean()*100, 2), '%')
print('Customers with full 90-day window:', cust.observable_90d.sum(), 'of', len(cust))

Item revenue = order revenue : True
Orders                       : 13295
Customers                    : 10000
Total net revenue (INR)      : 8418458
90-day repeat rate           : 22.0 %
Cart abandonment rate        : 72.58 %
Session -> order conversion  : 13.73 %
Blended CAC (acq. campaigns) : 637.65
Late delivery share          : 12.35 %
Customers with full 90-day window: 9495 of 10000


In [25]:
features = [
 ('orders','discount_pct','discount ÷ gross revenue','Shows how deep discounts cut into profit (Problem 1 and Q6)'),
 ('orders','net_revenue_after_refund_inr','net revenue − refund','Revenue the business actually keeps'),
 ('orders','contribution_margin_inr','net rev − refund − product cost − fulfilment − delivery − platform fee','Profit per order before marketing; revenue alone hides losses'),
 ('orders','contribution_incl_shipping_inr','contribution + shipping fee collected','Sensitivity check on how shipping is treated'),
 ('orders','contribution_margin_pct / gross_margin_pct','margin ÷ net revenue','Compare profitability across products and channels'),
 ('orders','delay_days, is_late','delivered date − promised date','Tests whether late delivery drives refunds (Q11)'),
 ('orders','is_refunded, has_discount, campaign_attributed','binary flags','Easy segmentation; shows attribution gaps'),
 ('orders','order_number, is_first_order, days_since_first_order','rank of order per customer','Defines new vs repeat purchase and reorder timing'),
 ('order_items','line_margin_inr, line_margin_pct','net line revenue − product cost','Shows which products earn profit, not just revenue (Q7)'),
 ('carts','discount_pct','offered discount ÷ cart value','Tests discount effect on conversion (Q6)'),
 ('carts','is_converted','1 if order_id exists','Cart-level conversion flag'),
 ('carts','abandon_stage','from checkout_started, payment_attempted, cart_status','Shows where carts are lost, so each stage gets its own fix (Problem 3)'),
 ('web_sessions','has_cart, reached_checkout, attempted_payment, converted','joined from carts','Builds the website funnel (Q3)'),
 ('web_sessions','is_mobile, load_speed_bucket','device and binned load time','Tests device and speed effects (Q4)'),
 ('web_sessions','is_anonymous, campaign_missing','null flags','Quantifies attribution gaps and anonymous traffic'),
 ('ad_spend','ctr, cpc_inr','clicks ÷ impressions; spend ÷ clicks','Ad efficiency (Q9)'),
 ('social','engagements, engagement_rate, link_click_rate','sum of likes/comments/shares/saves ÷ reach','Compares content performance (Q10)'),
 ('crm','crm_channel','campaign → channel (Email or WhatsApp)','message_type is content type, so channel must come from the campaign'),
 ('customers','repeat_90d','another order 1–90 days after first','Problem 2 KPI and Phase 5 target'),
 ('customers','observable_90d','first order + 90 days ≤ last date in data','Flags customers whose 90-day window is complete, for fair modelling'),
 ('customers','days_to_second_order, repeat_any','date difference','Reorder timing for CRM reminders'),
 ('customers','total_orders, total_net_revenue_inr, avg_order_value_inr, total_contribution_inr','aggregated from orders','Customer value (Q5 and CLV)'),
 ('customers','recency_days, tenure_days, cohort_month','dates vs snapshot date','Standard retention and churn inputs'),
 ('customers','acquisition_campaign, acquisition_channel(_group)','joined names','Customer quality by channel (Q2, Q5)'),
 ('customers','crm_delivered, crm_open_rate, crm_click_rate','aggregated from CRM','Links engagement to retention'),
 ('customers','web_sessions, abandoned_carts, avg_page_load_seconds','aggregated from sessions and carts','Behaviour signals for prediction'),
 ('campaign_month','roas, cac_inr, ctr','revenue ÷ spend; spend ÷ new customers; clicks ÷ impressions','Core KPIs for dashboard and Problem 1'),
 ('monthly','cac_inr (acquisition only)','acquisition spend ÷ new customers','Shows the ₹540 → ₹720 CAC trend'),
]
feature_dict = pd.DataFrame(features, columns=['table','feature','how_derived','why_it_matters'])
feature_dict.to_csv(PROC / 'feature_dictionary.csv', index=False)
feature_dict.head(10)

,table,feature,how_derived,why_it_matters
0,orders,discount_pct,discount ÷ gross revenue,Shows how deep discounts cut into profit (Prob...
1,orders,net_revenue_after_refund_inr,net revenue − refund,Revenue the business actually keeps
2,orders,contribution_margin_inr,net rev − refund − product cost − fulfilment −...,Profit per order before marketing; revenue alo...
3,orders,contribution_incl_shipping_inr,contribution + shipping fee collected,Sensitivity check on how shipping is treated
4,orders,contribution_margin_pct / gross_margin_pct,margin ÷ net revenue,Compare profitability across products and chan...
5,orders,"delay_days, is_late",delivered date − promised date,Tests whether late delivery drives refunds (Q11)
6,orders,"is_refunded, has_discount, campaign_attributed",binary flags,Easy segmentation; shows attribution gaps
7,orders,"order_number, is_first_order, days_since_first...",rank of order per customer,Defines new vs repeat purchase and reorder timing
8,order_items,"line_margin_inr, line_margin_pct",net line revenue − product cost,"Shows which products earn profit, not just rev..."
9,carts,discount_pct,offered discount ÷ cart value,Tests discount effect on conversion (Q6)


In [27]:
outputs = {'orders_clean': o, 'order_items_clean': li, 'customers_features': cust, 'sessions_clean': s,
           'carts_clean': c, 'ad_spend_clean': ad, 'campaign_month': cmth, 'monthly_summary': monthly,
           'crm_messages_clean': m, 'social_posts_clean': so, 'lead_events_clean': ld,
           'campaigns_clean': camp, 'products_clean': products, 'channels_clean': channels, 'calendar_clean': calendar}
for name, df in outputs.items():
    df.to_csv(PROC / f'{name}.csv', index=False)
    print(f'saved {name}.csv  {df.shape}')

saved orders_clean.csv  (13295, 34)
saved order_items_clean.csv  (17284, 13)
saved customers_features.csv  (10000, 40)
saved sessions_clean.csv  (76389, 23)
saved carts_clean.csv  (38260, 16)
saved ad_spend_clean.csv  (3160, 10)
saved campaign_month.csv  (105, 17)
saved monthly_summary.csv  (12, 7)
saved crm_messages_clean.csv  (23670, 12)
saved social_posts_clean.csv  (1260, 18)
saved lead_events_clean.csv  (7406, 9)
saved campaigns_clean.csv  (11, 9)
saved products_clean.csv  (8, 5)
saved channels_clean.csv  (9, 4)
saved calendar_clean.csv  (365, 7)
